# Potato Leaf Classification: Healthy vs Early Blight

**Dataset:** [PlantVillage (mohitsingh1804)](https://www.kaggle.com/datasets/mohitsingh1804/plantvillage) — pre-split `train/` and `test/` folders, each with `Healthy Potato` and `Potato Early Blight` subfolders.

## Read this before trusting any number below

1. **This is an easy binary cut.** The full potato subset of PlantVillage has 3 classes (Healthy, Early Blight, Late Blight). Restricting to Healthy vs Early Blight removes the genuinely hard confusion pair (Early vs Late Blight look similar; both look nothing like a healthy leaf). Expect high accuracy — that reflects the task's easiness, not model brilliance.
2. **PlantVillage has known leakage issues** — near-duplicate images (same leaf, rotated/cropped) can appear on both sides of a train/test split, inflating test accuracy. Section 3 below runs a perceptual-hash duplicate check between the provided train and test folders. **Look at that output before trusting the final test accuracy.**
3. **Lab-background bias.** These are controlled-background lab photos. A model trained here is not validated for real field photos (dirt, shadows, other leaves in frame, phone camera blur). Don't present this as field-deployment-ready without new field data.
4. We hold out a **validation split carved from `train/` only** — `test/` is touched exactly once, at final evaluation, so we don't quietly overfit our modeling decisions to it.

## Plan
- EDA + class balance
- Leakage / near-duplicate check (train vs test)
- Data pipeline with augmentation
- Model 1: Custom CNN (baseline)
- Model 2: Transfer learning (EfficientNetV2-B0)
- Evaluation: accuracy / precision / recall / F1 / ROC-AUC / PR-AUC / confusion matrix
- Grad-CAM explainability
- Misclassification review
- Robustness check (brightness / blur perturbations)
- Save best model


## 1. Setup

In [ ]:
import os, glob, random, hashlib, json, math, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 2
CLASS_NAMES = ["Healthy Potato", "Potato Early Blight"]


## 1b. Colab only — download the dataset via Kaggle API

Get a token first (one-time, ~30 seconds):
1. Go to kaggle.com → click your profile picture → **Settings**
2. Scroll to **API** → click **Create New Token** — this downloads a file called `kaggle.json`
3. Run the cell below, and when it prompts "Choose Files", upload that `kaggle.json`

Skip this cell entirely if you're on Kaggle instead of Colab (the dataset auto-mounts there).


In [ ]:
import os

if not os.path.isdir("./plantvillage") or not os.listdir("./plantvillage"):
    IN_COLAB = "google.colab" in str(get_ipython())
    if IN_COLAB:
        os.system("pip install -q kaggle")
        if not os.path.exists("/root/.kaggle/kaggle.json"):
            from google.colab import files
            print("Upload kaggle.json now:")
            uploaded = files.upload()
            os.makedirs("/root/.kaggle", exist_ok=True)
            # handle whatever filename was actually uploaded
            uploaded_name = list(uploaded.keys())[0]
            os.system(f"cp {uploaded_name} /root/.kaggle/kaggle.json")
            os.system("chmod 600 /root/.kaggle/kaggle.json")

        print("Downloading dataset...")
        ret = os.system(
            "kaggle datasets download -d mohitsingh1804/plantvillage -p ./plantvillage --unzip"
        )
        if ret != 0:
            print("!! Download failed. Common causes:")
            print("   - kaggle.json wasn't uploaded correctly (re-run this cell)")
            print("   - You haven't accepted the dataset's terms on kaggle.com yet")
            print("     (visit the dataset page and click 'Download' once manually)")
        else:
            print("Done. Contents of ./plantvillage:")
            print(os.listdir("./plantvillage"))
    else:
        print("Not running in Colab and ./plantvillage doesn't exist yet.")
        print("If you're on Kaggle, the dataset should be at /kaggle/input/ once attached via 'Add Input'.")
else:
    print("./plantvillage already populated, skipping download.")
    print(os.listdir("./plantvillage"))


In [ ]:
# Locate the dataset. Works on Kaggle (auto-mounted input) or a local download.
CANDIDATE_ROOTS = [
    "/kaggle/input/plantvillage",
    "/kaggle/input",
    "./plantvillage",
    "./data",
    ".",
]

def find_dataset_root():
    for root in CANDIDATE_ROOTS:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, _ in os.walk(root):
            names = {d.lower() for d in dirnames}
            if any("healthy" in n for n in names) or ("train" in names and "test" in names):
                return dirpath
    return None

DATA_ROOT = find_dataset_root()
print("Detected data root:", DATA_ROOT)

if DATA_ROOT is None:
    print("!! Dataset not found automatically. Here's what actually exists so you can set it manually:\n")
    for root in CANDIDATE_ROOTS:
        if os.path.isdir(root):
            print(f"--- contents of {root} ---")
            for dirpath, dirnames, filenames in os.walk(root):
                depth = dirpath[len(root):].count(os.sep)
                if depth > 3:
                    continue
                indent = "  " * depth
                print(f"{indent}{dirpath}/")
                for d in dirnames:
                    print(f"{indent}  [dir] {d}")
                if filenames and depth <= 3:
                    print(f"{indent}  ({len(filenames)} files)")
        else:
            print(f"--- {root} does not exist ---")
    print("\nMost likely cause: the PlantVillage dataset isn't attached to this notebook/kernel yet.")
    print("On Kaggle: click 'Add Input' (top right) and add mohitsingh1804/plantvillage, then re-run.")
    print("Locally/Colab: download+unzip the dataset and point DATA_ROOT at the folder that")
    print("directly contains 'train' and 'test' subfolders, e.g.:")
    print("   DATA_ROOT = \'/kaggle/input/plantvillage/PlantVillage\'")


In [ ]:
# --- If auto-detection above failed, set this manually and re-run ---
# DATA_ROOT = "/kaggle/input/plantvillage/<actual_subfolder>"

if DATA_ROOT is None:
    raise FileNotFoundError(
        "DATA_ROOT is None — the dataset wasn't found. See the diagnostic listing printed "
        "in the previous cell, set DATA_ROOT manually to the folder containing 'train' and "
        "'test' subfolders, and re-run this cell."
    )

TRAIN_DIR = os.path.join(DATA_ROOT, "train")
TEST_DIR = os.path.join(DATA_ROOT, "test")

def resolve_class_dir(split_dir, wanted):
    """Case/spacing-tolerant lookup of a class subfolder."""
    wanted_norm = wanted.lower().replace(" ", "").replace("_", "")
    for d in os.listdir(split_dir):
        if os.path.isdir(os.path.join(split_dir, d)):
            d_norm = d.lower().replace(" ", "").replace("_", "")
            if wanted_norm in d_norm or d_norm in wanted_norm:
                return os.path.join(split_dir, d)
    raise FileNotFoundError(f"Could not find a folder matching '{wanted}' in {split_dir}")

CLASS_DIRS = {
    "train": {c: resolve_class_dir(TRAIN_DIR, c) for c in CLASS_NAMES},
    "test":  {c: resolve_class_dir(TEST_DIR, c) for c in CLASS_NAMES},
}
CLASS_DIRS


## 2. EDA: class counts and sample images

In [ ]:
def list_images(folder):
    exts = (".jpg", ".jpeg", ".png", ".bmp")
    return [f for f in os.listdir(folder) if f.lower().endswith(exts)]

counts = []
for split in ["train", "test"]:
    for c in CLASS_NAMES:
        n = len(list_images(CLASS_DIRS[split][c]))
        counts.append({"split": split, "class": c, "count": n})

counts_df = pd.DataFrame(counts)
print(counts_df.pivot(index="class", columns="split", values="count"))

fig, ax = plt.subplots(figsize=(7,4))
sns.barplot(data=counts_df, x="class", y="count", hue="split", ax=ax)
ax.set_title("Image counts by class and split")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


In [ ]:
# Sample grid
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for row, split in enumerate(["train", "test"]):
    for col, c in enumerate(CLASS_NAMES):
        files = list_images(CLASS_DIRS[split][c])[:2]
        for i, f in enumerate(files):
            ax = axes[row, col*2 + i]
            img = Image.open(os.path.join(CLASS_DIRS[split][c], f))
            ax.imshow(img)
            ax.set_title(f"{split}/{c}", fontsize=9)
            ax.axis("off")
plt.tight_layout()
plt.show()


## 3. Leakage check — do NOT skip this

PlantVillage is known to contain near-duplicate images across splits. We hash every image
(perceptual hash, tolerant to minor recompression) and check for exact/near matches between
`train/` and `test/`. If a meaningful fraction of test images have a near-duplicate in train,
the test accuracy in Section 8 is inflated and should be reported with that caveat.


In [ ]:
def phash(path, hash_size=8):
    """Simple average-hash (fast, no extra deps beyond PIL/numpy)."""
    img = Image.open(path).convert("L").resize((hash_size, hash_size), Image.LANCZOS)
    arr = np.asarray(img, dtype=np.float64)
    avg = arr.mean()
    bits = (arr > avg).flatten()
    return "".join("1" if b else "0" for b in bits)

def hamming(a, b):
    return sum(c1 != c2 for c1, c2 in zip(a, b))

def hash_split(split):
    out = {}
    for c in CLASS_NAMES:
        d = CLASS_DIRS[split][c]
        for f in list_images(d):
            out[(split, c, f)] = phash(os.path.join(d, f))
    return out

train_hashes = hash_split("train")
test_hashes = hash_split("test")
print(f"Hashed {len(train_hashes)} train images, {len(test_hashes)} test images")


In [ ]:
# Near-duplicate search: for each test image, is there a train image within a small
# Hamming distance (<=5 bits out of 64)? This is O(n*m) — fine at PlantVillage's scale
# per-class, but subsample if this is slow on your machine.
THRESHOLD = 5

train_items = list(train_hashes.items())
leak_hits = []
for (split, c, f), h in test_hashes.items():
    for (tsplit, tc, tf), th in train_items:
        if tc != c:
            continue
        if hamming(h, th) <= THRESHOLD:
            leak_hits.append({"test_file": f, "test_class": c, "train_match": tf, "distance": hamming(h, th)})
            break

leak_df = pd.DataFrame(leak_hits)
leak_rate = len(leak_df) / max(len(test_hashes), 1)
print(f"Possible near-duplicates found: {len(leak_df)} / {len(test_hashes)} test images ({leak_rate:.1%})")
if len(leak_df):
    display(leak_df.head(10))
if leak_rate > 0.05:
    print("!! WARNING: >5% of test images have a near-duplicate in train.")
    print("   Test accuracy below is likely inflated. Consider a hash-deduplicated re-split.")
else:
    print("Leakage rate looks low — test accuracy is more trustworthy.")


## 4. Train / validation split

We carve a validation set out of `train/` (stratified, 85/15) so `test/` remains a truly
held-out set touched only once, at final evaluation. This protects us from unconsciously
tuning architecture/hyperparameters against the test set.


In [ ]:
from sklearn.model_selection import train_test_split

def build_filepaths_df(split):
    rows = []
    for label, c in enumerate(CLASS_NAMES):
        d = CLASS_DIRS[split][c]
        for f in list_images(d):
            rows.append({"filepath": os.path.join(d, f), "label": label, "class": c})
    return pd.DataFrame(rows)

train_all_df = build_filepaths_df("train")
test_df = build_filepaths_df("test")

train_df, val_df = train_test_split(
    train_all_df, test_size=0.15, stratify=train_all_df["label"], random_state=SEED
)

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))
print(train_df["class"].value_counts())


## 5. tf.data pipelines

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess(filepath, label, augment=False):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMG_SIZE)
    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.random_brightness(img, 0.15)
        img = tf.image.random_contrast(img, 0.85, 1.15)
        img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

def make_dataset(df, augment=False, shuffle=False, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices((df["filepath"].values, df["label"].values))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED)
    ds = ds.map(lambda fp, lb: load_and_preprocess(fp, lb, augment=augment), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, augment=True, shuffle=True)
val_ds = make_dataset(val_df, augment=False, shuffle=False)
test_ds = make_dataset(test_df, augment=False, shuffle=False)

# class weights, in case of imbalance
from sklearn.utils.class_weight import compute_class_weight
class_weights_arr = compute_class_weight("balanced", classes=np.array([0,1]), y=train_df["label"].values)
class_weights = {0: class_weights_arr[0], 1: class_weights_arr[1]}
print("Class weights:", class_weights)


## 6. Model 1 — Custom CNN (baseline)

In [ ]:
def build_custom_cnn(input_shape=(224,224,3)):
    inputs = keras.Input(shape=input_shape)
    x = inputs
    for filters in [32, 64, 128, 256]:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inputs, outputs, name="custom_cnn")

custom_cnn = build_custom_cnn()
custom_cnn.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc"), keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall")],
)
custom_cnn.summary()


In [ ]:
callbacks_cnn = [
    keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
    keras.callbacks.ModelCheckpoint("custom_cnn_best.keras", monitor="val_auc", mode="max", save_best_only=True),
]

history_cnn = custom_cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=class_weights,
    callbacks=callbacks_cnn,
)


In [ ]:
def plot_history(history, title):
    hist = history.history
    fig, axes = plt.subplots(1, 2, figsize=(12,4))
    axes[0].plot(hist["loss"], label="train")
    axes[0].plot(hist["val_loss"], label="val")
    axes[0].set_title(f"{title} — loss"); axes[0].legend()
    axes[1].plot(hist["accuracy"], label="train")
    axes[1].plot(hist["val_accuracy"], label="val")
    axes[1].set_title(f"{title} — accuracy"); axes[1].legend()
    plt.tight_layout(); plt.show()

plot_history(history_cnn, "Custom CNN")


## 7. Model 2 — Transfer learning (EfficientNetV2-B0)

Two-phase training: freeze the backbone and train the head first, then unfreeze the
top layers and fine-tune at a low learning rate.


In [ ]:
def build_transfer_model(input_shape=(224,224,3), base_trainable=False):
    base = keras.applications.EfficientNetV2B0(
        include_top=False, weights="imagenet", input_shape=input_shape
    )
    base.trainable = base_trainable
    inputs = keras.Input(shape=input_shape)
    x = keras.applications.efficientnet_v2.preprocess_input(inputs * 255.0)  # our pipeline already scales to [0,1]
    x = base(x, training=base_trainable)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs, name="efficientnetv2b0_transfer")
    return model, base

transfer_model, backbone = build_transfer_model(base_trainable=False)
transfer_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc"), keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall")],
)
transfer_model.summary()


In [ ]:
# Phase 1: train head only
callbacks_head = [
    keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5, restore_best_weights=True),
]

history_head = transfer_model.fit(
    train_ds, validation_data=val_ds, epochs=10,
    class_weight=class_weights, callbacks=callbacks_head,
)


In [ ]:
# Phase 2: unfreeze top of backbone, fine-tune at low LR
backbone.trainable = True
FINE_TUNE_AT = len(backbone.layers) - 30  # unfreeze last ~30 layers
for layer in backbone.layers[:FINE_TUNE_AT]:
    layer.trainable = False

transfer_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc"), keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall")],
)

callbacks_ft = [
    keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7),
    keras.callbacks.ModelCheckpoint("transfer_model_best.keras", monitor="val_auc", mode="max", save_best_only=True),
]

history_ft = transfer_model.fit(
    train_ds, validation_data=val_ds, epochs=20,
    class_weight=class_weights, callbacks=callbacks_ft,
)


In [ ]:
plot_history(history_head, "EfficientNetV2-B0 — head training")
plot_history(history_ft, "EfficientNetV2-B0 — fine-tuning")


## 8. Evaluation on the held-out TEST set

Touched for the first time here. Report both models side by side. Cross-reference
against Section 3's leakage check before trusting these numbers at face value.


In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)

def evaluate_model(model, ds, df, name):
    y_true = df["label"].values
    y_prob = model.predict(ds, verbose=0).ravel()
    y_pred = (y_prob >= 0.5).astype(int)

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
    }
    return metrics, y_true, y_pred, y_prob

results_cnn, yt_cnn, yp_cnn, yprob_cnn = evaluate_model(custom_cnn, test_ds, test_df, "Custom CNN")
results_tl, yt_tl, yp_tl, yprob_tl = evaluate_model(transfer_model, test_ds, test_df, "EfficientNetV2-B0")

results_df = pd.DataFrame([results_cnn, results_tl]).set_index("model")
results_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11,4.5))
for ax, (yt, yp, name) in zip(axes, [(yt_cnn, yp_cnn, "Custom CNN"), (yt_tl, yp_tl, "EfficientNetV2-B0")]):
    cm = confusion_matrix(yt, yp)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f"Confusion matrix — {name}")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.show()

print("Custom CNN report:\n", classification_report(yt_cnn, yp_cnn, target_names=CLASS_NAMES))
print("EfficientNetV2-B0 report:\n", classification_report(yt_tl, yp_tl, target_names=CLASS_NAMES))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11,4.5))
for yt, yprob, name in [(yt_cnn, yprob_cnn, "Custom CNN"), (yt_tl, yprob_tl, "EfficientNetV2-B0")]:
    fpr, tpr, _ = roc_curve(yt, yprob)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(yt, yprob):.3f})")
    prec, rec, _ = precision_recall_curve(yt, yprob)
    axes[1].plot(rec, prec, label=name)
axes[0].plot([0,1],[0,1],'k--', alpha=0.3)
axes[0].set_title("ROC curve"); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].legend()
axes[1].set_title("Precision-Recall curve"); axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].legend()
plt.tight_layout(); plt.show()


**Pick the model with the better ROC-AUC / PR-AUC on test, not just raw accuracy** — with any
class imbalance, accuracy is the easiest metric to game and the least informative.

In [ ]:
BEST_NAME = results_df["roc_auc"].idxmax()
best_model = custom_cnn if BEST_NAME == "Custom CNN" else transfer_model
print("Best model by test ROC-AUC:", BEST_NAME)


## 9. Grad-CAM — is the model actually looking at the lesions?

In [ ]:
def get_last_conv_layer(model):
    for layer in reversed(model.layers):
        if isinstance(layer, layers.Conv2D):
            return layer.name
        # for nested models (e.g. transfer learning backbone)
        if hasattr(layer, "layers"):
            for sublayer in reversed(layer.layers):
                if isinstance(sublayer, layers.Conv2D):
                    return sublayer.name
    return None

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = 0
        class_channel = predictions[:, pred_index]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

# Try to locate a conv layer inside the best model (handles nested backbone models too)
try:
    last_conv = get_last_conv_layer(best_model)
    print("Using conv layer:", last_conv)

    sample_rows = test_df[test_df["class"] == "Potato Early Blight"].sample(4, random_state=SEED)
    fig, axes = plt.subplots(2, 4, figsize=(14,7))
    for i, (_, row) in enumerate(sample_rows.iterrows()):
        img = Image.open(row["filepath"]).convert("RGB").resize(IMG_SIZE)
        arr = np.expand_dims(np.array(img)/255.0, axis=0).astype(np.float32)
        heatmap = make_gradcam_heatmap(arr, best_model, last_conv)
        heatmap_resized = np.array(Image.fromarray((heatmap*255).astype(np.uint8)).resize(IMG_SIZE))

        axes[0, i].imshow(img); axes[0, i].axis("off"); axes[0, i].set_title("Original")
        axes[1, i].imshow(img); axes[1, i].imshow(heatmap_resized, cmap="jet", alpha=0.45); axes[1, i].axis("off")
        axes[1, i].set_title("Grad-CAM")
    plt.tight_layout(); plt.show()
except Exception as e:
    print("Grad-CAM failed (common with some nested transfer-learning graphs):", e)
    print("If this fails on the transfer model, re-run this cell with best_model = custom_cnn.")


## 10. Misclassification review

In [ ]:
y_prob_best = yprob_tl if BEST_NAME == "EfficientNetV2-B0" else yprob_cnn
y_pred_best = yp_tl if BEST_NAME == "EfficientNetV2-B0" else yp_cnn

test_df_reset = test_df.reset_index(drop=True)
wrong_idx = np.where(y_pred_best != test_df_reset["label"].values)[0]
print(f"{len(wrong_idx)} / {len(test_df_reset)} test images misclassified by {BEST_NAME}")

if len(wrong_idx):
    show_idx = wrong_idx[:8]
    fig, axes = plt.subplots(2, 4, figsize=(14,7))
    for ax, idx in zip(axes.flatten(), show_idx):
        row = test_df_reset.iloc[idx]
        img = Image.open(row["filepath"])
        ax.imshow(img); ax.axis("off")
        ax.set_title(f"true={row['class'][:12]}\npred_prob={y_prob_best[idx]:.2f}", fontsize=8)
    plt.tight_layout(); plt.show()


## 11. Robustness check

Real photos aren't studio-lit. Quick sanity check: how much does test accuracy degrade
under mild blur and brightness/contrast shifts? If accuracy collapses under small
perturbations, the model is likely leaning on background/lighting artifacts rather than
lesion shape — consistent with the "lab-background bias" warning at the top.


In [ ]:
import scipy.ndimage as ndi

def perturb_batch(images, kind):
    imgs = images.numpy()
    if kind == "blur":
        out = np.stack([ndi.gaussian_filter(im, sigma=(1.5,1.5,0)) for im in imgs])
    elif kind == "dark":
        out = np.clip(imgs * 0.6, 0, 1)
    elif kind == "bright":
        out = np.clip(imgs * 1.4, 0, 1)
    else:
        out = imgs
    return tf.convert_to_tensor(out, dtype=tf.float32)

def eval_under_perturbation(model, ds, kind):
    y_true_all, y_pred_all = [], []
    for images, labels in ds:
        p_images = perturb_batch(images, kind) if kind != "none" else images
        probs = model.predict(p_images, verbose=0).ravel()
        y_pred_all.extend((probs >= 0.5).astype(int))
        y_true_all.extend(labels.numpy())
    return accuracy_score(y_true_all, y_pred_all)

robustness_rows = []
for kind in ["none", "blur", "dark", "bright"]:
    acc = eval_under_perturbation(best_model, test_ds, kind)
    robustness_rows.append({"perturbation": kind, "accuracy": acc})

robustness_df = pd.DataFrame(robustness_rows)
robustness_df


In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
sns.barplot(data=robustness_df, x="perturbation", y="accuracy", ax=ax)
ax.set_ylim(0, 1)
ax.set_title(f"{BEST_NAME} — accuracy under perturbation")
plt.tight_layout(); plt.show()

drop = robustness_df.loc[robustness_df['perturbation']=='none','accuracy'].values[0] - robustness_df['accuracy'].min()
if drop > 0.10:
    print(f"!! Accuracy drops by {drop:.1%} under some perturbation — model is not robust to lighting/blur shifts.")
else:
    print(f"Max accuracy drop under perturbation: {drop:.1%} — reasonably robust to these mild shifts.")


## 12. Save the best model

In [ ]:
best_model.save("potato_blight_best_model.keras")
with open("test_metrics.json", "w") as f:
    json.dump(results_df.to_dict(orient="index"), f, indent=2)
print("Saved potato_blight_best_model.keras and test_metrics.json")


## 13. Honest conclusions

Fill this in after running, but the structure should cover:
- Which model won on test ROC-AUC/PR-AUC (not just accuracy), and by how much.
- **What Section 3 found.** If leakage rate was non-trivial, say so explicitly and discount
  the test accuracy accordingly instead of reporting it uncritically.
- **What Section 11 found.** If robustness collapsed under mild perturbation, say the model
  is validated for lab-condition images only, not field deployment.
- Explicit next step if this were to become a real tool: re-collect a field-condition test
  set (phone photos, natural backgrounds) and re-evaluate before claiming it works in practice.
